In [ ]:
#@title ⚙️ Settings: paste your platform details, pick a dataset
PLATFORM_URL = ""   #@param {type:"string"}
SUBMIT_TOKEN = ""   #@param {type:"string"}
DATASET = "quickdraw"  #@param ["quickdraw", "fashion", "emoji", "mnist"]
QUICKDRAW_CATEGORIES = "cat,pizza,rocket,smiley face"  #@param {type:"string"}
QUICK = False  #@param {type:"boolean"}

# 🎨 Make the machine draw: DCGAN → WGAN-GP → Conditional GAN

Today you build three image generators from scratch and train them on real data:

| Part | Model | Big idea |
|---|---|---|
| 1 | **DCGAN** | Generator vs discriminator, a two-player game ⚔️ |
| 2 | **WGAN-GP** | A *critic* + gradient penalty → a loss curve that actually means something 📉 |
| 3 | **Conditional GAN** | Tell the generator *what* to draw 🎯 |
| 4 | **Controllable generation** | Style × class grids, class morphs, latent walks ✨ |

**How it works**
1. Pick a dataset in the ⚙️ settings cell above (and paste `PLATFORM_URL` + `SUBMIT_TOKEN` from the platform's challenge page).
2. `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
3. Fill in every `# TODO` (look for `...`). The ✅ check cell right below each one tells you if you nailed it.
4. The last cell submits your W-distance + your best image. The instructor ⭐ stars the coolest generations!

**Datasets** (all images become 32×32, so one set of model code handles all of them):
- `quickdraw` ✏️ doodles from [Google Quick, Draw!](https://github.com/googlecreativelab/quickdraw-dataset), pick your own categories (names from [this list](https://github.com/googlecreativelab/quickdraw-dataset/blob/master/categories.txt)). Licence: **CC BY 4.0**, © Google.
- `fashion` 👟 [Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist) by Zalando Research (MIT licence).
- `emoji` 😎 [OpenMoji](https://openmoji.org) colour emoji (3 channels!). Licence: **CC BY-SA 4.0**, © OpenMoji / HfG Schwäbisch Gmünd. Small dataset (~1.6k images), so we train for more epochs and use random flips.
- `mnist` 🔢 the classic handwritten digits (Yann LeCun & Corinna Cortes, CC BY-SA 3.0).

`QUICK = True` trains on a tiny subset for 1 epoch so the whole notebook runs in a few minutes on a CPU. The images will look like noise. That's fine, it's for testing!

In [ ]:
#@title 🧰 Setup: imports, device, constants
import io, json, math, os, time, urllib.parse, urllib.request, zipfile
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.utils import make_grid

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("🖥️  device:", device)
if device.type != "cuda":
    print("⚠️  No GPU found. Training will be slow. In Colab: Runtime → Change runtime type → T4 GPU"
          + ("" if QUICK else "  (or set QUICK = True to just test the notebook)"))

torch.manual_seed(0)
np.random.seed(0)
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
IMG_SIZE = 32          # every dataset is padded/resized to 32×32
QUICK_MAX = 2000       # QUICK mode: train on at most this many images

## 1️⃣ Load the data

Every dataset ends up as the same thing:
- `X`: float tensor `(N, CHANNELS, 32, 32)` scaled to **[-1, 1]** (to match the generator's `tanh` output)
- `Y`: class index per image, `CLASS_NAMES[Y]` is the human name
- `CHANNELS` (1 = grayscale, 3 = colour) and `N_CLASSES`

28×28 sources (MNIST, Fashion, Quick Draw) get 2 pixels of padding on each side → 32×32. That makes the conv stack super clean: **1 → 4 → 8 → 16 → 32**.

In [ ]:
#@title 📦 Load the dataset
def download(url, path, max_bytes=None):
    """Download once and cache. max_bytes: only fetch the start of the file."""
    if os.path.exists(path):
        return path
    headers = {"Range": f"bytes=0-{max_bytes - 1}"} if max_bytes else {}
    with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=300) as r:
        data = r.read(max_bytes) if max_bytes else r.read()
    with open(path + ".part", "wb") as f:
        f.write(data)
    os.replace(path + ".part", path)
    return path


def pad28(imgs_uint8, fill=0):
    """(N, 28, 28) uint8 → (N, 1, 32, 32) uint8"""
    t = torch.as_tensor(np.asarray(imgs_uint8)).unsqueeze(1)
    return F.pad(t, (2, 2, 2, 2), value=fill)


def load_torchvision(name):
    from torchvision import datasets
    ds = {"mnist": datasets.MNIST, "fashion": datasets.FashionMNIST}[name](DATA_DIR, train=True, download=True)
    names = [c.split(" - ")[0] if name == "mnist" else c.split("/")[0].lower() for c in ds.classes]
    return pad28(ds.data.numpy()), ds.targets.clone(), names


QD_URL = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/"

def load_quickdraw(categories, per_class):
    """Each category's .npy is ~100 MB, so we only download the first `per_class` drawings (HTTP Range)."""
    xs, ys = [], []
    for k, cat in enumerate(categories):
        path = f"{DATA_DIR}/quickdraw_{cat.replace(' ', '_')}_{per_class}.npypart"
        try:
            download(QD_URL + urllib.parse.quote(cat) + ".npy", path, max_bytes=1024 + per_class * 784)
        except urllib.error.HTTPError as e:
            raise ValueError(f"Quick, Draw! has no category {cat!r} (HTTP {e.code}). Check the spelling in QUICKDRAW_CATEGORIES.") from e
        with open(path, "rb") as f:
            version = np.lib.format.read_magic(f)
            read_header = np.lib.format.read_array_header_1_0 if version == (1, 0) else np.lib.format.read_array_header_2_0
            shape, _, _ = read_header(f)
            raw = f.read()
        n = min(per_class, shape[0], len(raw) // 784)
        imgs = np.frombuffer(raw, dtype=np.uint8, count=n * 784).reshape(n, 28, 28)
        xs.append(255 - pad28(imgs))        # black ink on white paper
        ys.append(torch.full((n,), k, dtype=torch.long))
        print(f"  ✏️  {cat}: {n} drawings")
    return torch.cat(xs), torch.cat(ys), list(categories)


OPENMOJI_ZIP = "https://github.com/hfg-gmuend/openmoji/releases/download/17.0.0/openmoji-72x72-color.zip"
OPENMOJI_JSON = "https://raw.githubusercontent.com/hfg-gmuend/openmoji/17.0.0/data/openmoji.json"
# OpenMoji "group" → our class. Skin-tone variants, flags, components and the extra
# non-standard icons are skipped (they'd swamp everything else); tiny groups are merged.
EMOJI_CLASSES = {
    "smileys-emotion": "smileys", "people-body": "people", "animals-nature": "animals",
    "food-drink": "food", "travel-places": "travel", "activities": "objects",
    "objects": "objects", "symbols": "symbols",
}

def load_emoji():
    from PIL import Image
    meta = json.load(open(download(OPENMOJI_JSON, f"{DATA_DIR}/openmoji.json"), encoding="utf-8"))
    zf = zipfile.ZipFile(download(OPENMOJI_ZIP, f"{DATA_DIR}/openmoji-72x72-color.zip"))
    files = set(zf.namelist())
    names = list(dict.fromkeys(EMOJI_CLASSES.values()))          # unique, keeps order
    xs, ys = [], []
    for e in meta:
        if e["group"] not in EMOJI_CLASSES or e["skintone"] or e["hexcode"] + ".png" not in files:
            continue
        im = Image.open(io.BytesIO(zf.read(e["hexcode"] + ".png"))).convert("RGBA")
        bg = Image.new("RGBA", im.size, (255, 255, 255, 255))
        bg.alpha_composite(im)                                        # transparent → white
        im = bg.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
        xs.append(torch.as_tensor(np.array(im)).permute(2, 0, 1))
        ys.append(names.index(EMOJI_CLASSES[e["group"]]))
    return torch.stack(xs), torch.tensor(ys), names


t0 = time.time()
if DATASET in ("mnist", "fashion"):
    X, Y, CLASS_NAMES = load_torchvision(DATASET)
elif DATASET == "quickdraw":
    cats = [c.strip() for c in QUICKDRAW_CATEGORIES.split(",") if c.strip()]
    assert cats, "Pick some Quick, Draw! categories, separated by commas"
    if len(cats) == 1:
        print("💡 Only one category: the conditional GAN part is more fun with 2+ (e.g. 'cat,pizza,rocket')")
    X, Y, CLASS_NAMES = load_quickdraw(cats, per_class=QUICK_MAX if QUICK else 15000)
elif DATASET == "emoji":
    X, Y, CLASS_NAMES = load_emoji()
else:
    raise ValueError(f"Unknown DATASET {DATASET!r}")

if QUICK and len(X) > QUICK_MAX:                     # random subset so every class shows up
    keep = torch.randperm(len(X))[:QUICK_MAX]
    X, Y = X[keep], Y[keep]

X = X.float() / 127.5 - 1.0                           # uint8 [0, 255] → float [-1, 1]
N, CHANNELS = X.shape[0], X.shape[1]
N_CLASSES = len(CLASS_NAMES)
FLIP = DATASET == "emoji"                             # small dataset → random horizontal flips
assert X.shape[2:] == (IMG_SIZE, IMG_SIZE)
print(f"📦 {DATASET}: {N} images, {CHANNELS} channel(s), {IMG_SIZE}×{IMG_SIZE}, "
      f"{N_CLASSES} classes {CLASS_NAMES}  ({time.time() - t0:.0f}s)")
print("   images per class:", dict(zip(CLASS_NAMES, torch.bincount(Y, minlength=N_CLASSES).tolist())))

In [ ]:
#@title 👀 Look at some real images
def to_img(t):
    """(B, C, H, W) in [-1, 1] → displayable [0, 1]"""
    return (t.detach().float().cpu() * 0.5 + 0.5).clamp(0, 1)


def grid_np(imgs, nrow):
    return make_grid(to_img(imgs), nrow=nrow, padding=2, pad_value=1).permute(1, 2, 0).numpy()


def show_grid(imgs, nrow=8, title=None, ax=None, size=6):
    if ax is None:
        _, ax = plt.subplots(figsize=(size, size * math.ceil(len(imgs) / nrow) / nrow + 0.4))
    ax.imshow(grid_np(imgs, nrow), interpolation="nearest")
    ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


shown = list(range(min(N_CLASSES, 10)))
fig, axes = plt.subplots(len(shown), 1, figsize=(8, 1.05 * len(shown)))
axes = np.atleast_1d(axes)
for ax, k in zip(axes, shown):
    idx = (Y == k).nonzero().view(-1)[:10]
    ax.imshow(grid_np(X[idx], nrow=10), interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_ylabel(CLASS_NAMES[k], rotation=0, ha="right", va="center", fontsize=11)
fig.suptitle(f"Real {DATASET} images", y=1.0)
plt.tight_layout()
plt.show()

## 2️⃣ DCGAN: the original two-player game ⚔️

- **Generator** `G(z)`: random noise `z` (100 numbers) → fake image. It **upsamples** with `ConvTranspose2d`.
- **Discriminator** `D(x)`: image → one **logit** ("how real does this look?"). It **downsamples** with strided `Conv2d`.

DCGAN rules (Radford et al., 2015): strided convs instead of pooling, BatchNorm, ReLU in G, LeakyReLU(0.2) in D, `tanh` at G's output, Adam(2e-4, β₁=0.5), weights ~ N(0, 0.02).

### 🧩 TODO: get the shapes right (`conv_shapes`)
The output size of a conv layer:

$$\text{conv: } out = \left\lfloor \frac{in + 2\cdot pad - kernel}{stride} \right\rfloor + 1 \qquad\qquad \text{transposed conv: } out = (in - 1)\cdot stride - 2\cdot pad + kernel$$

1. Code both formulas.
2. Pick `(kernel, stride, padding)` for the four kinds of layers so that **G goes 1 → 4 → 8 → 16 → 32** and **D goes 32 → 16 → 8 → 4 → 1**. Kernel 4 everywhere works nicely 😉

In [ ]:
def conv_out(size, kernel, stride=1, pad=0):
    return ...  # TODO (use // for floor division)

def conv_transpose_out(size, kernel, stride=1, pad=0):
    return ...  # TODO

#           (kernel, stride, padding)
G_FIRST = (..., ..., ...)   # TODO  ConvTranspose2d: 1×1 → 4×4  (noise "pixel" → tiny image)
UP      = (..., ..., ...)   # TODO  ConvTranspose2d: doubles the size  4 → 8 → 16 → 32
DOWN    = (..., ..., ...)   # TODO  Conv2d: halves the size  32 → 16 → 8 → 4
D_LAST  = (..., ..., ...)   # TODO  Conv2d: 4×4 → 1×1 (one logit per image)

# ---- nothing to change below: the models use your tuples ----
Z_DIM = 100
G_CH = D_CH = 32 if QUICK else 64      # QUICK: slimmer nets so a CPU can cope

class Generator(nn.Module):
    """z (B, z_dim) → image (B, out_ch, 32, 32) in [-1, 1]"""
    def __init__(self, z_dim=Z_DIM, out_ch=CHANNELS, ch=G_CH):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, ch * 4, *G_FIRST, bias=False),   # 1  → 4
            nn.BatchNorm2d(ch * 4), nn.ReLU(True),
            nn.ConvTranspose2d(ch * 4, ch * 2, *UP, bias=False),       # 4  → 8
            nn.BatchNorm2d(ch * 2), nn.ReLU(True),
            nn.ConvTranspose2d(ch * 2, ch, *UP, bias=False),           # 8  → 16
            nn.BatchNorm2d(ch), nn.ReLU(True),
            nn.ConvTranspose2d(ch, out_ch, *UP),                       # 16 → 32
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z.view(z.size(0), -1, 1, 1))


class Discriminator(nn.Module):
    """image (B, in_ch, 32, 32) → one logit/score per image (B,).  batchnorm=False gives the WGAN critic."""
    def __init__(self, in_ch=CHANNELS, ch=D_CH, batchnorm=True):
        super().__init__()
        norm = nn.BatchNorm2d if batchnorm else (lambda c: nn.Identity())
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, ch, *DOWN), nn.LeakyReLU(0.2, True),                                    # 32 → 16
            nn.Conv2d(ch, ch * 2, *DOWN, bias=not batchnorm), norm(ch * 2), nn.LeakyReLU(0.2, True),      # 16 → 8
            nn.Conv2d(ch * 2, ch * 4, *DOWN, bias=not batchnorm), norm(ch * 4), nn.LeakyReLU(0.2, True),  # 8 → 4
            nn.Conv2d(ch * 4, 1, *D_LAST),                                                           # 4 → 1
        )

    def forward(self, x):
        return self.net(x).view(-1)


def init_weights(m):
    """DCGAN init: N(0, 0.02) conv weights, N(1, 0.02) BatchNorm scales."""
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

In [ ]:
#@title ✅ Check: shapes
try:
    assert conv_out(32, 4, 2, 1) == 16 and conv_out(29, 4, 2, 1) == 14 and conv_out(7, 7, 1, 0) == 1, \
        "conv_out is off. out = (in + 2*pad - kernel) // stride + 1"
    assert conv_transpose_out(1, 4, 1, 0) == 4 and conv_transpose_out(16, 4, 2, 1) == 32 and conv_transpose_out(8, 5, 2, 2) == 15, \
        "conv_transpose_out is off. out = (in - 1)*stride - 2*pad + kernel"
    s, g_sizes = 1, []
    for kind in [G_FIRST, UP, UP, UP]:
        s = conv_transpose_out(s, *kind); g_sizes.append(s)
    s, d_sizes = 32, []
    for kind in [DOWN, DOWN, DOWN, D_LAST]:
        s = conv_out(s, *kind); d_sizes.append(s)
    print("G sizes: 1 →", " → ".join(map(str, g_sizes)), "   D sizes: 32 →", " → ".join(map(str, d_sizes)))
    assert g_sizes == [4, 8, 16, 32], "Generator should go 1 → 4 → 8 → 16 → 32. Check G_FIRST and UP."
    assert d_sizes == [16, 8, 4, 1], "Discriminator should go 32 → 16 → 8 → 4 → 1. Check DOWN and D_LAST."
    _G, _D = Generator(), Discriminator()
    _fake = _G(torch.randn(3, Z_DIM))
    assert _fake.shape == (3, CHANNELS, IMG_SIZE, IMG_SIZE), f"G output {tuple(_fake.shape)}"
    assert _D(_fake).shape == (3,), f"D output {tuple(_D(_fake).shape)}, expected one logit per image"
except (TypeError, RuntimeError, AttributeError) as e:
    raise AssertionError(f"Some `...` blanks are still empty (or not numbers) 👀  [{e}]") from e
print(f"G: {sum(p.numel() for p in _G.parameters())/1e6:.2f}M params,  D: {sum(p.numel() for p in _D.parameters())/1e6:.2f}M params")
print("✅ nice, the shapes line up!")

### 🧩 TODO: the GAN losses (`gan_losses`)
D outputs **logits**, so we use `F.binary_cross_entropy_with_logits` (stable even for huge logits, unlike `log(sigmoid(l))`).

- **Discriminator**: real images → target **1**, fakes → target **0**. Add the two losses.
- **Generator** (the *non-saturating* trick): instead of minimising `log(1 − D(G(z)))` (whose gradient vanishes when D wins), G uses BCE on its fakes with target **1**: "make D call my fakes real".

Hint: `torch.ones_like(logits)` / `torch.zeros_like(logits)`.

In [ ]:
bce = F.binary_cross_entropy_with_logits

def discriminator_loss(real_logits, fake_logits):
    return ...  # TODO: BCE(real → 1) + BCE(fake → 0)

def generator_loss(fake_logits):
    return ...  # TODO: non-saturating, BCE(fake → 1)

In [ ]:
#@title ✅ Check: GAN losses
def _ref_bce(l, t):
    return (l.clamp(min=0) - l * t + torch.log1p(torch.exp(-l.abs()))).mean()

_g = torch.Generator().manual_seed(4)
_real, _fake = torch.randn(16, generator=_g) * 3, torch.randn(16, generator=_g) * 3
_d, _gl = discriminator_loss(_real, _fake), generator_loss(_fake)
assert torch.is_tensor(_d) and torch.is_tensor(_gl), "Return torch tensors (scalars), not `...`"
assert _d.dim() == 0 and _gl.dim() == 0, "Losses should be scalars (use the mean, which BCE does by default)"
assert torch.allclose(_d, _ref_bce(_real, 1) + _ref_bce(_fake, 0), atol=1e-5), "discriminator_loss: real → target 1, fake → target 0, then add"
assert not torch.allclose(_gl, -_ref_bce(_fake, 0), atol=1e-4), "That's the *saturating* G loss. Use BCE with target 1 instead"
assert torch.allclose(_gl, _ref_bce(_fake, 1), atol=1e-5), "generator_loss: BCE on fake logits with target 1"
_ext = discriminator_loss(torch.tensor([1000., -1000.]), torch.tensor([-1000., 1000.]))
assert torch.isfinite(_ext), "Loss blows up for extreme logits. Use F.binary_cross_entropy_with_logits"
print(f"confident D: D loss {discriminator_loss(torch.full((4,), 4.), torch.full((4,), -4.)):.3f}, "
      f"G loss {generator_loss(torch.full((4,), -4.)):.3f}  (G is losing → big G loss, big gradient)")
print("✅ nice, losses look right!")

In [ ]:
#@title 🛠️ Training helpers (batches on the GPU, sample strips, curves)
X_dev, Y_dev = X.to(device), Y.to(device)

def batches(batch_size):
    """One epoch of shuffled (images, labels) batches, all on the device. Emoji get random flips."""
    perm = torch.randperm(N, device=device)
    for i in range(0, N - batch_size + 1, batch_size):
        idx = perm[i:i + batch_size]
        x = X_dev[idx]
        if FLIP:
            flip = torch.rand(x.size(0), 1, 1, 1, device=device) < 0.5
            x = torch.where(flip, x.flip(3), x)
        yield x, Y_dev[idx]


def n_epochs(target_iters, batch_size):
    """Same number of training steps for every dataset (small datasets get more epochs)."""
    if QUICK:
        return 1
    return max(1, round(target_iters / max(1, N // batch_size)))


def snapshot_epochs(epochs, k=6):
    return sorted(set(np.linspace(1, epochs, min(epochs, k)).round().astype(int).tolist()))


def show_progress(snaps, nrow=8, title=""):
    fig, axes = plt.subplots(1, len(snaps), figsize=(2.6 * len(snaps), 2.9), squeeze=False)
    for ax, (ep, imgs) in zip(axes[0], snaps):
        show_grid(imgs, nrow=nrow, ax=ax, title=f"epoch {ep}")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def smooth(v, k=25):
    v = np.asarray(v, dtype=float)
    if len(v) < k:
        return v
    return np.convolve(v, np.ones(k) / k, mode="valid")


def plot_curves(curves, title, xlabel="step"):
    plt.figure(figsize=(8, 3.5))
    for label, v in curves.items():
        plt.plot(v, alpha=0.25)
        plt.plot(np.arange(len(smooth(v))) + (len(v) - len(smooth(v))) // 2, smooth(v), label=label, color=plt.gca().lines[-1].get_color())
    plt.xlabel(xlabel); plt.title(title); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
#@title 🏋️ Train the DCGAN
DC_BATCH, DC_LR = 128, 2e-4
DC_EPOCHS = n_epochs(5000, DC_BATCH)

torch.manual_seed(1)
G = Generator().to(device).apply(init_weights)
D = Discriminator().to(device).apply(init_weights)
opt_G = torch.optim.Adam(G.parameters(), lr=DC_LR, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=DC_LR, betas=(0.5, 0.999))

fixed_z = torch.randn(32, Z_DIM, device=device)        # same noise every epoch → watch it evolve
dc_curves = {"D loss": [], "G loss": [], "D(real)": [], "D(fake)": []}
dc_snaps, show_at = [], snapshot_epochs(DC_EPOCHS)
t0 = time.time()
for epoch in range(1, DC_EPOCHS + 1):
    for real, _ in batches(DC_BATCH):
        b = real.size(0)
        # 1) Discriminator: push D(real) → 1 and D(fake) → 0
        fake = G(torch.randn(b, Z_DIM, device=device))
        d_real, d_fake = D(real), D(fake.detach())      # detach: don't update G here
        loss_D = discriminator_loss(d_real, d_fake)
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()
        # 2) Generator: make D call the fakes real
        loss_G = generator_loss(D(fake))
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()

        dc_curves["D loss"].append(loss_D.item()); dc_curves["G loss"].append(loss_G.item())
        dc_curves["D(real)"].append(torch.sigmoid(d_real).mean().item())
        dc_curves["D(fake)"].append(torch.sigmoid(d_fake).mean().item())
    if epoch in show_at:
        G.eval()
        with torch.no_grad():
            dc_snaps.append((epoch, G(fixed_z).cpu()))
        G.train()
        print(f"epoch {epoch:3d}/{DC_EPOCHS}  D loss {np.mean(dc_curves['D loss'][-50:]):.3f}  "
              f"G loss {np.mean(dc_curves['G loss'][-50:]):.3f}  D(real) {np.mean(dc_curves['D(real)'][-50:]):.2f}  "
              f"D(fake) {np.mean(dc_curves['D(fake)'][-50:]):.2f}  [{time.time() - t0:.0f}s]")

show_progress(dc_snaps, title=f"DCGAN on {DATASET}: same noise, epoch by epoch")
plot_curves({k: dc_curves[k] for k in ["D loss", "G loss"]}, "DCGAN losses (they fight, they don't converge)")

🤔 **Notice:** the DCGAN losses bounce around and don't tell you much about image quality. And if D gets too good, G's gradients can vanish → mode collapse (every sample looks the same). Part 2 fixes both.

## 3️⃣ WGAN-GP: a loss that actually means something 📉

Swap the classifier for a **critic** `C(x)` that outputs an unbounded **score** (no sigmoid, no log). The gap in mean scores estimates the **Wasserstein (earth-mover) distance** between real and fake images:

$$W \approx \text{mean}(C(\text{real})) - \text{mean}(C(\text{fake}))$$

- the **critic** wants W **big** → it minimises `−W`
- the **generator** wants its fakes to score high → it minimises `−mean(C(fake))`

That estimate is only valid if the critic is **1-Lipschitz** (gradient norm ≤ 1). WGAN-GP (Gulrajani et al., 2017) softly enforces it on random points **between** real and fake images:

$$\hat{x} = \varepsilon\cdot\text{real} + (1-\varepsilon)\cdot\text{fake},\quad \varepsilon\sim U[0,1] \qquad GP = \text{mean}\Big(\big(\lVert\nabla_{\hat{x}} C(\hat{x})\rVert_2 - 1\big)^2\Big)$$

Critic loss = `−W + 10·GP`. Other WGAN-GP changes: **no BatchNorm** in the critic (GP is per-image, BatchNorm mixes images), **n_critic = 5** critic steps per G step, Adam(1e-4, betas=(0, 0.9)).

### 🧩 TODO: `wgan_losses` + `gradient_penalty`
`torch.autograd.grad(outputs, inputs, grad_outputs=..., create_graph=True)` gives ∂outputs/∂inputs. `create_graph=True` matters: the GP is part of the loss, so PyTorch must be able to backprop *through the gradient* into the critic's weights.

In [ ]:
GP_WEIGHT = 10.0

def wasserstein_estimate(real_scores, fake_scores):
    return ...  # TODO

def critic_loss(real_scores, fake_scores):
    return ...  # TODO: critic wants W big (GP is added in the training loop)

def wgan_generator_loss(fake_scores):
    return ...  # TODO

def gradient_penalty(critic, real, fake):
    eps = torch.rand(real.size(0), 1, 1, 1, device=real.device)        # one ε per image
    x_hat = ...  # TODO 1: points between real and fake (then .requires_grad_(True))
    scores = critic(x_hat)
    grads = torch.autograd.grad(outputs=scores, inputs=x_hat, grad_outputs=torch.ones_like(scores),
                                create_graph=...)[0]                    # TODO 2: True or False?
    norms = ...  # TODO 3: L2 norm of each image's gradient → shape (B,)   hint: grads.view(B, -1).norm(2, dim=1)
    return ...   # TODO 4: mean of (norm − 1)²

In [ ]:
#@title ✅ Check: Wasserstein losses + gradient penalty
_r, _f = torch.tensor([3.0, 2.5, 4.0]), torch.tensor([-1.0, 0.5, -2.0])
try:
    assert torch.isclose(torch.as_tensor(wasserstein_estimate(_r, _f)), torch.tensor(4.0)), "W = mean(real scores) − mean(fake scores)"
    assert torch.isclose(torch.as_tensor(critic_loss(_r, _f)), torch.tensor(-4.0)), "critic_loss = −W (the critic maximises W)"
    assert torch.isclose(torch.as_tensor(wgan_generator_loss(_f)), torch.tensor(0.8333), atol=1e-3), "G loss = −mean(fake scores)"

    class _Lin(nn.Module):              # C(x) = <w, x>  →  ∇C = w everywhere, so GP = (‖w‖ − 1)²
        def __init__(self, w): super().__init__(); self.w = nn.Parameter(w)
        def forward(self, x): return (x * self.w).flatten(1).sum(1)
    _real, _fake = torch.randn(8, 1, 2, 2), torch.randn(8, 1, 2, 2)
    _gp = gradient_penalty(_Lin(torch.full((1, 2, 2), 1.5)), _real, _fake)       # ‖w‖ = 3
    assert torch.is_tensor(_gp) and _gp.dim() == 0, "gradient_penalty should return a scalar tensor"
    assert torch.isclose(_gp, torch.tensor(4.0), atol=1e-4), f"GP = {_gp.item():.3f}, expected (3 − 1)² = 4. Norm per image, not over the whole batch?"
    assert torch.isclose(gradient_penalty(_Lin(torch.full((1, 2, 2), 0.5)), _real, _fake), torch.tensor(0.0), atol=1e-5)
    assert _gp.requires_grad, "GP has no gradient path to the critic's weights. Use create_graph=True"

    # C(x) = ½‖x‖² → ∇C = x̂. With real = 2 and fake = 0 the answer depends on ε: only-real gives 9, only-fake gives 1
    _q = lambda x: 0.5 * (x ** 2).flatten(1).sum(1)
    _gp2 = gradient_penalty(_q, torch.full((512, 1, 2, 2), 2.0), torch.zeros(512, 1, 2, 2)).item()
    assert 1.5 < _gp2 < 3.3, f"GP = {_gp2:.2f}. x̂ should be a random mix ε·real + (1 − ε)·fake (expected ≈ 2.3)"
except (TypeError, RuntimeError, AttributeError) as e:
    raise AssertionError(f"Some `...` blanks are still empty 👀  [{e}]") from e
print("✅ nice, the critic will stay (roughly) 1-Lipschitz!")

In [ ]:
#@title 🏋️ Train the WGAN-GP (critic: no BatchNorm, n_critic = 5)
WG_BATCH, WG_LR, N_CRITIC = 64, 1e-4, 5
WG_EPOCHS = n_epochs(15000, WG_BATCH)           # 15000 critic steps = 3000 generator steps

torch.manual_seed(2)
G_w = Generator().to(device).apply(init_weights)
C = Discriminator(batchnorm=False).to(device).apply(init_weights)      # the critic
opt_Gw = torch.optim.Adam(G_w.parameters(), lr=WG_LR, betas=(0.0, 0.9))
opt_C = torch.optim.Adam(C.parameters(), lr=WG_LR, betas=(0.0, 0.9))

wg_curves = {"W-distance estimate": [], "gradient penalty": [], "G loss": []}
wg_snaps, show_at, step = [], snapshot_epochs(WG_EPOCHS), 0
t0 = time.time()
for epoch in range(1, WG_EPOCHS + 1):
    for real, _ in batches(WG_BATCH):
        b = real.size(0)
        # 1) Critic step (every batch)
        fake = G_w(torch.randn(b, Z_DIM, device=device)).detach()
        c_real, c_fake = C(real), C(fake)
        gp = gradient_penalty(C, real, fake)
        loss_C = critic_loss(c_real, c_fake) + GP_WEIGHT * gp
        opt_C.zero_grad(); loss_C.backward(); opt_C.step()
        wg_curves["W-distance estimate"].append(wasserstein_estimate(c_real, c_fake).item())
        wg_curves["gradient penalty"].append(gp.item())
        step += 1
        # 2) Generator step (once every N_CRITIC critic steps)
        if step % N_CRITIC == 0:
            loss_G = wgan_generator_loss(C(G_w(torch.randn(b, Z_DIM, device=device))))
            opt_Gw.zero_grad(); loss_G.backward(); opt_Gw.step()
            wg_curves["G loss"].append(loss_G.item())
    if epoch in show_at:
        G_w.eval()
        with torch.no_grad():
            wg_snaps.append((epoch, G_w(fixed_z).cpu()))
        G_w.train()
        print(f"epoch {epoch:3d}/{WG_EPOCHS}  W-distance {np.mean(wg_curves['W-distance estimate'][-100:]):.3f}  "
              f"GP {np.mean(wg_curves['gradient penalty'][-100:]):.3f}  [{time.time() - t0:.0f}s]")

w = wg_curves["W-distance estimate"]
W_FINAL = float(np.mean(w[-max(1, len(w) // 10):]))      # average over the last 10% of critic steps
print(f"\n📏 final W-distance estimate: {W_FINAL:.3f}  (lower = fakes are closer to the real data)")

In [ ]:
#@title 📈 WGAN-GP results
show_progress(wg_snaps, title=f"WGAN-GP on {DATASET}: same noise, epoch by epoch")
plot_curves({"W-distance estimate": wg_curves["W-distance estimate"]},
            "WGAN-GP: the W-distance estimate should FALL as images get better", xlabel="critic step")
plot_curves({"gradient penalty": wg_curves["gradient penalty"]}, "Gradient penalty (should stay small)", xlabel="critic step")

## 4️⃣ Conditional GAN: tell it *what* to draw 🎯

A plain GAN draws *something*; you can't choose what. A conditional GAN (Mirza & Osindero, 2014) gives the **class label** to **both** networks:

- **Generator**: concatenate noise `z (B, Z)` with the one-hot label `(B, K)` → `(B, Z + K)`
- **Discriminator**: turn the one-hot label into `K` constant 32×32 **label maps** and stack them onto the image channels: `(B, C, 32, 32)` + `(B, K, 32, 32)` → `(B, C + K, 32, 32)`

Now D rejects fakes that look real but show the **wrong class**, so G has to obey the label.

### 🧩 TODO: `condition_inputs`
Useful: `F.one_hot(labels, K).float()`, `torch.cat(..., dim=1)`, and `y.view(B, K, 1, 1).expand(B, K, H, W)`.

In [ ]:
def one_hot(labels, n_classes=N_CLASSES):
    return ...  # TODO: (B,) int → (B, n_classes) float

def generator_input(z, y_onehot):
    return ...  # TODO: (B, Z) + (B, K) → (B, Z + K)

def discriminator_input(images, y_onehot):
    B, _, H, W = images.shape
    K = y_onehot.size(1)
    maps = ...  # TODO: label maps (B, K, H, W), every pixel of map k = y_onehot[:, k]
    return ...  # TODO: (B, C, H, W) + (B, K, H, W) → (B, C + K, H, W)

# ---- nothing to change below ----
class CondGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.g = Generator(z_dim=Z_DIM + N_CLASSES)            # same DCGAN generator, bigger input
    def forward(self, z, y_onehot):
        return self.g(generator_input(z, y_onehot))

class CondDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.d = Discriminator(in_ch=CHANNELS + N_CLASSES)     # same DCGAN discriminator, extra channels
    def forward(self, images, y_onehot):
        return self.d(discriminator_input(images, y_onehot))

In [ ]:
#@title ✅ Check: conditioning inputs
try:
    _y = one_hot(torch.tensor([0, N_CLASSES - 1, 1 % N_CLASSES]))
    assert torch.is_tensor(_y) and _y.shape == (3, N_CLASSES), f"one_hot shape should be (3, {N_CLASSES})"
    assert _y.dtype == torch.float32 and torch.equal(_y.sum(1), torch.ones(3)) and _y[1, -1] == 1, "one_hot: float, a single 1 per row"
    _gi = generator_input(torch.zeros(3, 7), _y)
    assert _gi.shape == (3, 7 + N_CLASSES), f"generator_input shape {tuple(_gi.shape)}, expected (3, {7 + N_CLASSES})"
    assert torch.equal(_gi[:, 7:], _y), "generator_input: z first, then the label"
    _imgs = torch.randn(3, CHANNELS, 5, 6)
    _di = discriminator_input(_imgs, _y)
    assert _di.shape == (3, CHANNELS + N_CLASSES, 5, 6), f"discriminator_input shape {tuple(_di.shape)}"
    assert torch.equal(_di[:, :CHANNELS], _imgs), "discriminator_input: image channels first"
    assert torch.equal(_di[:, CHANNELS:], _y[:, :, None, None].expand(3, N_CLASSES, 5, 6)), "label map k should be filled with y_onehot[:, k]"
    _cg, _cd = CondGenerator(), CondDiscriminator()
    _x = _cg(torch.randn(3, Z_DIM), _y)
    assert _x.shape == (3, CHANNELS, IMG_SIZE, IMG_SIZE) and _cd(_x, _y).shape == (3,)
except (TypeError, RuntimeError, AttributeError) as e:
    raise AssertionError(f"Some `...` blanks are still empty 👀  [{e}]") from e
print("✅ nice, both networks now know the label!")

In [ ]:
#@title 🏋️ Train the conditional GAN
CG_BATCH, CG_LR = 128, 2e-4
CG_EPOCHS = n_epochs(5000, CG_BATCH)

torch.manual_seed(3)
G_c = CondGenerator().to(device).apply(init_weights)
D_c = CondDiscriminator().to(device).apply(init_weights)
opt_Gc = torch.optim.Adam(G_c.parameters(), lr=CG_LR, betas=(0.5, 0.999))
opt_Dc = torch.optim.Adam(D_c.parameters(), lr=CG_LR, betas=(0.5, 0.999))

# fixed z per row, one class per column → every snapshot is already a style × class grid
fixed_zc = torch.randn(4, 1, Z_DIM, device=device).expand(-1, N_CLASSES, -1).reshape(-1, Z_DIM)
fixed_yc = one_hot(torch.arange(N_CLASSES, device=device).repeat(4))
cg_curves = {"D loss": [], "G loss": []}
cg_snaps, show_at = [], snapshot_epochs(CG_EPOCHS)
t0 = time.time()
for epoch in range(1, CG_EPOCHS + 1):
    for real, labels in batches(CG_BATCH):
        y = one_hot(labels)
        fake = G_c(torch.randn(real.size(0), Z_DIM, device=device), y)
        loss_D = discriminator_loss(D_c(real, y), D_c(fake.detach(), y))   # "real image OF class y?"
        opt_Dc.zero_grad(); loss_D.backward(); opt_Dc.step()
        loss_G = generator_loss(D_c(fake, y))
        opt_Gc.zero_grad(); loss_G.backward(); opt_Gc.step()
        cg_curves["D loss"].append(loss_D.item()); cg_curves["G loss"].append(loss_G.item())
    if epoch in show_at:
        G_c.eval()
        with torch.no_grad():
            cg_snaps.append((epoch, G_c(fixed_zc, fixed_yc).cpu()))
        G_c.train()
        print(f"epoch {epoch:3d}/{CG_EPOCHS}  D loss {np.mean(cg_curves['D loss'][-50:]):.3f}  "
              f"G loss {np.mean(cg_curves['G loss'][-50:]):.3f}  [{time.time() - t0:.0f}s]")

show_progress(cg_snaps, nrow=N_CLASSES, title=f"Conditional GAN: columns = {', '.join(CLASS_NAMES)}")
plot_curves(cg_curves, "Conditional GAN losses")

## 5️⃣ Controllable generation ✨

The trained cGAN has two knobs: the **label** picks the class, the **noise z** picks the style. Moving smoothly between two settings is **interpolation**:

$$\text{lerp}(a, b, t) = (1-t)\cdot a + t\cdot b,\qquad t\in[0,1]$$

- `lerp_path(a, b, steps)`: `steps` evenly spaced points from `a` to `b` → shape `(steps, D)`
- `label_morph(from_class, to_class, steps)`: a path between two one-hot labels. Feeding these **soft labels** to G morphs one class into another while the style stays fixed 🫠

### 🧩 TODO: `controllable`

In [ ]:
def lerp_path(a, b, steps):
    """a, b: (D,) → (steps, D); first row = a, last row = b"""
    t = torch.linspace(0, 1, steps, device=a.device).unsqueeze(1)    # (steps, 1)
    return ...  # TODO

def label_morph(from_class, to_class, steps, n_classes=N_CLASSES):
    """soft labels sliding from one class to another → (steps, n_classes)"""
    return ...  # TODO: lerp_path between two one-hot vectors (hint: torch.eye(n_classes))

In [ ]:
#@title ✅ Check: interpolation
try:
    _a, _b = torch.tensor([0.0, 2.0, -1.0]), torch.tensor([4.0, 2.0, 1.0])
    _p = lerp_path(_a, _b, 5)
    assert torch.is_tensor(_p) and _p.shape == (5, 3), "lerp_path should return shape (steps, D)"
    assert torch.allclose(_p[0], _a) and torch.allclose(_p[-1], _b), "first row = a, last row = b"
    assert torch.allclose(_p[2], torch.tensor([2.0, 2.0, 0.0])), "the middle row should be halfway"
    _m = label_morph(0, 1, 3, n_classes=4)
    assert _m.shape == (3, 4), "label_morph shape should be (steps, n_classes)"
    assert torch.allclose(_m, torch.tensor([[1., 0, 0, 0], [0.5, 0.5, 0, 0], [0, 1., 0, 0]])), "label_morph(0, 1, 3) should go [1,0,..] → [.5,.5,..] → [0,1,..]"
except (TypeError, RuntimeError, AttributeError) as e:
    raise AssertionError(f"Some `...` blanks are still empty 👀  [{e}]") from e
print("✅ nice, let's play with the knobs!")

In [ ]:
#@title 🎛️ Style × class grid, class morph, latent walk → result.png
MORPH_FROM = 0  #@param {type:"integer"}
MORPH_TO = 1  #@param {type:"integer"}
WALK_CLASS = 0  #@param {type:"integer"}
MORPH_FROM, MORPH_TO, WALK_CLASS = MORPH_FROM % N_CLASSES, MORPH_TO % N_CLASSES, WALK_CLASS % N_CLASSES
STEPS = 10
G_c.eval()
torch.manual_seed(7)
with torch.no_grad():
    # 1) style × class: each row = one fixed z (style), each column = one class
    z = torch.randn(8, 1, Z_DIM, device=device).expand(-1, N_CLASSES, -1).reshape(-1, Z_DIM)
    style_class = G_c(z, one_hot(torch.arange(N_CLASSES, device=device).repeat(8))).cpu()
    # 2) class morph: same z per row, the label slides from MORPH_FROM to MORPH_TO
    y = label_morph(MORPH_FROM, MORPH_TO, STEPS).to(device)
    morph = torch.cat([G_c(torch.randn(1, Z_DIM, device=device).expand(STEPS, -1), y) for _ in range(3)]).cpu()
    # 3) latent walk: same label, z slides between two random points
    y = one_hot(torch.full((STEPS,), WALK_CLASS, device=device))
    walk = torch.cat([G_c(lerp_path(*torch.randn(2, Z_DIM, device=device), STEPS), y) for _ in range(3)]).cpu()

fig = plt.figure(figsize=(12, 8.2))
gs = fig.add_gridspec(2, 2, width_ratios=[max(N_CLASSES, 4), STEPS], height_ratios=[1, 1])
ax = fig.add_subplot(gs[:, 0])
show_grid(style_class, nrow=N_CLASSES, ax=ax, title="style (rows) × class (columns)")
step_px = IMG_SIZE + 2
ax.axis("on"); ax.set_yticks([])
ax.set_xticks([2 + step_px * k + IMG_SIZE / 2 for k in range(N_CLASSES)])
ax.set_xticklabels(CLASS_NAMES, rotation=45 if N_CLASSES > 5 else 0, ha="right" if N_CLASSES > 5 else "center")
for s in ax.spines.values():
    s.set_visible(False)
show_grid(morph, nrow=STEPS, ax=fig.add_subplot(gs[0, 1]),
          title=f"class morph: {CLASS_NAMES[MORPH_FROM]} → {CLASS_NAMES[MORPH_TO]}")
show_grid(walk, nrow=STEPS, ax=fig.add_subplot(gs[1, 1]), title=f"latent walk: {CLASS_NAMES[WALK_CLASS]}")
fig.suptitle(f"Conditional GAN on {DATASET}" + ("  (QUICK mode)" if QUICK else ""), fontsize=14)
plt.tight_layout()
plt.savefig("result.png", dpi=100, bbox_inches="tight")
plt.show()
size_mb = os.path.getsize("result.png") / 1e6
if size_mb > 1.5:                                  # keep the upload small
    fig.savefig("result.png", dpi=60, bbox_inches="tight")
    size_mb = os.path.getsize("result.png") / 1e6
print(f"💾 saved result.png ({size_mb:.2f} MB)")

🎨 **Play time:** change `MORPH_FROM`, `MORPH_TO` and `WALK_CLASS` (class indices: see the printout above) and rerun the cell. Which classes morph nicely? Which ones turn into weird hybrids?

## 6️⃣ Submit 🚀
Sends your final **W-distance** estimate from the WGAN-GP and your `result.png` to the platform. The instructor judges the images by eye and ⭐ stars the best ones.

In [ ]:
#@title 🚀 Submit your result to DL Lab
import base64, json, urllib.request, urllib.error
CHALLENGE_ID = "gans"   # "sequences" | "attention" | "gans"

def submit(metric_name=None, metric=None, image_path=None, notes=""):
    if not PLATFORM_URL or not SUBMIT_TOKEN:
        print("⚠️ Paste PLATFORM_URL and SUBMIT_TOKEN into the settings cell at the top (copy them from the platform's challenge page).")
        return
    body = {"dataset": DATASET, "metric_name": metric_name, "metric": None if metric is None else float(metric), "notes": notes}
    if image_path:
        with open(image_path, "rb") as f:
            body["image_b64"] = base64.b64encode(f.read()).decode()
    req = urllib.request.Request(PLATFORM_URL.rstrip("/") + f"/api/challenges/{CHALLENGE_ID}/submit",
                                 data=json.dumps(body).encode(), method="POST",
                                 headers={"Content-Type": "application/json", "Authorization": "Bearer " + SUBMIT_TOKEN.strip()})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            print("🎉", json.loads(r.read())["message"])
    except urllib.error.HTTPError as e:
        print("❌ Submit failed:", e.code, e.read().decode()[:300])
    except Exception as e:
        print("❌ Could not reach the platform:", e)

submit(metric_name="w_distance", metric=W_FINAL, image_path="result.png",
       notes=f"{DATASET}: classes={','.join(CLASS_NAMES)}; epochs DCGAN {DC_EPOCHS} / WGAN-GP {WG_EPOCHS} / cGAN {CG_EPOCHS}"
             + ("; QUICK" if QUICK else ""))